# ME5413: Autonomous Mobile Robot  

## Homework 1: Perception  
-- by Mai Jingyang, GitHub @Whitbrunn, please contact maij@u.nus.edu for any questions.

## Task 2 Multi Object Prediction

### 2.1 Basic Function Definitions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os 
from tqdm import tqdm

In [2]:
# Func. for drawing
def draw_road(road_polylines):
    for polyline in road_polylines:
            map_type = polyline[0,6]
            if map_type == 6:
                plt.plot(polyline[:, 0], polyline[:, 1], 'w', linestyle='dashed', linewidth=1)
            elif map_type == 7:
                plt.plot(polyline[:, 0], polyline[:, 1], 'w', linestyle='solid', linewidth=1)
            elif map_type == 8:
                plt.plot(polyline[:, 0], polyline[:, 1], 'w', linestyle='solid', linewidth=1)
            elif map_type == 9:
                plt.plot(polyline[:, 0], polyline[:, 1], 'xkcd:yellow', linestyle='dashed', linewidth=1)
            elif map_type == 10:
                plt.plot(polyline[:, 0], polyline[:, 1], 'xkcd:yellow', linestyle='dashed', linewidth=1)
            elif map_type == 11:
                plt.plot(polyline[:, 0], polyline[:, 1], 'xkcd:yellow', linestyle='solid', linewidth=1)
            elif map_type == 12:
                plt.plot(polyline[:, 0], polyline[:, 1], 'xkcd:yellow', linestyle='solid', linewidth=1)
            elif map_type == 13:
                plt.plot(polyline[:, 0], polyline[:, 1], 'xkcd:yellow', linestyle='dotted', linewidth=1)
            elif map_type == 15:
                plt.plot(polyline[:, 0], polyline[:, 1], 'k', linewidth=1)
            elif map_type == 16:
                plt.plot(polyline[:, 0], polyline[:, 1], 'k', linewidth=1)

def draw_rect(agent, type="others"):
    color = "dimgrey"
    if type=="others":
        pass
    elif type == "sdc":
        color = "purple"
    elif type == "agents_tp":
        color = "green"

    cenx = agent[0]
    ceny = agent[1]
    width = agent[3]
    height = agent[4] 
    vx = agent[7]
    vy = agent[8]

    # plt.plot(cenx,ceny, "bo", markersize=5)
    x_left = cenx - width / 2
    y_bottom = ceny - height / 2

    angle = np.arctan2(vy, vx) * 180 / np.pi

    ax = plt.gca()
    rect = plt.Rectangle((x_left, y_bottom), width, height, 
                             angle=angle, rotation_point="center" ,edgecolor="black", 
                             facecolor=color, linewidth=0.75, zorder=3)
    ax.add_patch(rect)
    
    

    # if color != "dimgrey":
    #     arrow_scale = 1.5 
    #     ax.quiver(cenx, ceny, vx*arrow_scale, vy*arrow_scale, angles='xy', scale_units='xy', scale=1, 
    #     color='blue', width=0.005, zorder=4)

def draw_traj(p_lst, color, label):
    x,y = zip(*p_lst)
    plt.plot(x,y, linestyle="-", linewidth=2, color = color, label = label, alpha = 0.7, zorder=5)
    # plt.plot(x,y, linestyle="-", linewidth=2, color = color, alpha = 0.7, zorder=5)
    if len(p_lst) > 1:
        x_start, y_start = p_lst[-2]  # 倒数第二个点
        x_end, y_end = p_lst[-1]      # 终点
        dx, dy = x_end - x_start, y_end - y_start  # 计算方向向量
        
        # 添加箭头（使用 annotate）
        plt.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                     arrowprops=dict(arrowstyle='-|>', color=color, lw=2))

def draw_gt_traj(inarray, color, label):
    x = []
    y = []
    for array in inarray:
        if array[9] != 0.:
            x.append(array[0])
            y.append(array[1])
        else:
            pass
    plt.plot(x,y, linestyle=":", linewidth=1, color = color, label = label, alpha = 0.7, zorder=4)
    # plt.plot(x,y, linestyle="-", linewidth=2, color = color, alpha = 0.7, zorder=5)
    if len(inarray) > 1:
        x_start = x[-2]
        y_start = y[-2]  # 倒数第二个点
        x_end = x[-1]
        y_end = y[-1]      # 终点
        dx, dy = x_end - x_start, y_end - y_start  # 计算方向向量
        
        # 添加箭头（使用 annotate）
        plt.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start), alpha = 0.7,
                     arrowprops=dict(arrowstyle='-|>', color=color, lw=1))        

In [21]:
# Func. for calculating prediction and errors
from math import sqrt

def read_state_info(input_state):
    x0 = input_state[0]
    y0 = input_state[1]
    v_x = input_state[7]
    v_y = input_state[8]
    return (x0, y0, v_x, v_y)

def is_state_valid(input_state):
    if input_state[9] != 0.:
        return True
    else:
        return False

# constant velocity
def constant_v_pred(input_state, t_span):
    x0, y0, v_x, v_y = read_state_info(input_state)

    pred_plst = []

    for t in range(t_span*10): # t=3,5 or 8s
        x_p = v_x*(t+1)*0.1 + x0
        y_p = v_y*(t+1)*0.1 + y0
        pred_plst.append((x_p,y_p))
    
    return pred_plst

# constant acceleration
def constant_a_pred(input_state1, input_state2, t_span):
    _, _, v_x_0, v_y_0 = read_state_info(input_state1)
    x0, y0, v_x, v_y = read_state_info(input_state2)

    ax = (v_x-v_x_0)/0.1
    ay = (v_y-v_y_0)/0.1

    pred_plst = []

    for t in range(t_span*10):
        x1 = 0.5*ax*(0.1*(t+1))**2 + v_x*0.1*(t+1) + x0
        y1 = 0.5*ay*(0.1*(t+1))**2 + v_y*0.1*(t+1) + y0

        pred_plst.append((x1,y1))
    return pred_plst


def calculate_ADE(traj1, gt):
    ADE_score = 0
    for i in range(len(gt)):
        p1 = traj1[i]
        p2 = gt[i] # p2[0] is cen_x, p2[1] is cen_y
        if p2[9] != 0.:
            ADE_score += sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
        else:
            pass
    return ADE_score/len(traj1)
    

def calculate_FDE(p1,p2):
    return sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)


### 2.2 Load Data

In [4]:
data_path = 'task2_data'
file_count = len(os.listdir(data_path))

print(f"The number of scenarios under {data_path} is: {file_count}")

The number of scenarios under task2_data is: 287


### 2.3 Q1: Constant Velocity Model Prediction

Use a constant velocity model to predict the future trajectories of the target agents at 3s, 5s, and 8s. Then, calculate the Average Displacement Error (ADE) and Final Displacement Error (FDE). The results should be averaged across all target agents and scenarios.

#### 2.3.1 Necessary function definitions

In [5]:
def cvm_pred_pre_agent(input_agent, curr_t, type):
    pred_name_lst =["3", "5", "8"]
    color_lst1 = ["darksalmon", "coral", "orangered"]
    color_lst2 = ["lightsteelblue", "cornflowerblue", "royalblue"]

    if type == "sdc_agent":
        color_lst = color_lst1
    elif type == "agents_tp":
        color_lst = color_lst2

    val_t = curr_t
    while is_state_valid(input_agent[val_t]) is not True:
        val_t +=1
    input_agent_curr_state = input_agent[val_t]


    agent_v_pred_dic = {}
    for i, name in enumerate(pred_name_lst[::-1]):
        pred = constant_v_pred(input_agent_curr_state,int(name))
        draw_traj(pred, color_lst[i], f"{type}-{name}s")
        
        predict_horizon = int(int(name)*10)
        gt = input_agent[val_t:min(val_t+predict_horizon, len(input_agent))]

        ADE_score = round(calculate_ADE(pred,gt), 4)
        end_id = -1
        while gt[end_id][9] == 0.:
            end_id -=1
        FDE_score = round(calculate_FDE(pred[end_id],gt[end_id]),4)
        

        # print(f"{name}s: ADE={ADE_score}, FDE={FDE_score}")
            
        agent_v_pred_dic[name] = [pred, ADE_score, FDE_score]
    return agent_v_pred_dic

def cvm_pred_pre_scenario(road_polylines,
                          all_agent_current, agents_to_predict_current_state,
                          sdc_current_state,
                          agents_to_predict,
                          sdc_agent,
                          curr_t,
                          scenario_id):
    plt.figure(figsize=(8,8))
    ax = plt.gca()
    fig = plt.gcf()
    fig.set_facecolor('xkcd:grey') 
    ax.set_facecolor('xkcd:grey')


    draw_road(road_polylines)

    for agent in all_agent_current:
        draw_rect(agent)

    for agent in agents_to_predict_current_state:
        draw_rect(agent,"agents_tp")

    
    
    for agent in agents_to_predict:
        agent_gt = agent[curr_t:curr_t+80]
        draw_gt_traj(agent_gt, "gold", "gt")

    agents_pred_lst = []
    for agent in agents_to_predict:
        agents_pred_lst.append(cvm_pred_pre_agent(agent, curr_t, "agents_tp"))


    ax.axis([-70+ sdc_current_state[0], 70+ sdc_current_state[0], -70+ sdc_current_state[1], 70 + sdc_current_state[1]])

    pred_name_lst =["3", "5", "8"]

    ADE_lst = np.array([0,0,0])
    FDE_lst = np.array([0,0,0])
    for agent_dic in agents_pred_lst:
        for i, name in enumerate(pred_name_lst):
            ADE_lst[i] += agent_dic[name][1]
            FDE_lst[i] += agent_dic[name][2]
    ADE_per_scenario = ADE_lst/len(agents_pred_lst)
    FDE_per_scenario = FDE_lst/len(agents_pred_lst)

    
    output_lst = []
    for i, name in enumerate(pred_name_lst):
        output_lst.append(f"{name}s: ADE={round(ADE_per_scenario[i],4)}, FDE={round(FDE_per_scenario[i],4)}")

    

    plt.figtext(0.75, 0.1, f"Average result of agents_tp in this scenario,\n"
                f"{output_lst[0]};\n"
                f"{output_lst[1]};\n"
                f"{output_lst[2]}.",
                ha='center',fontsize=10, bbox=dict(facecolor='white', alpha=0.5, edgecolor='gray'))
    
    plt.subplots_adjust(bottom=0.25)
    
    hands, las = ax.get_legend_handles_labels()
    hands_n = hands[::-1][:3]
    las_n = las[::-1][:3]
    hands_n.append(hands[0])
    las_n.append(las[0])

    ax.legend(hands_n,las_n,loc="upper left")

    plt.title("Constant Velocity Model Trajectory Prediction")
    filename = f'visualization/CVM/{scenario_id}.png' 
    plt.savefig(filename)       
    plt.close()

    return (ADE_per_scenario, FDE_per_scenario)


#### 2.3.2 Visualisation

Note:
- All 287 scenarios will be stored in png images in folder `task2_prediction/visualization/CVM`.
- In each image,
    - `agents_to_predict` are represented in green blocks, whose predicted trajectories are drawn in blue line gradually becoming lighter over time and the ground truth trajectories of both the above agents are represented in gold dotted line.
    - Other agents are represented in gray blocks.
    - The ADE & FDE scores across all predicted trajectories of `agents_to_predict` w.r.t 3s, 5s, 8s are shown in the right bottom of the image.

In [6]:
# Code for Q1
ADE_CVM_lst = [[],[],[]]
FDE_CVM_lst = [[],[],[]]

for file in tqdm(os.listdir(data_path)):

    info = np.load(data_path+'/'+file,allow_pickle=True)
    all_agent_trajs        = info['all_agent']    
    
    obj_types              = info['object_type']
    lane_polylines         = info['lane']           #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    road_polylines         = info['road_polylines'] #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    crosswalk_polylines    = info['crosswalk']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    speed_bump_polylines   = info['speed_bump']     #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    stop_signs_polylines   = info['stop_sign']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    drive_way_polylines    = info['drive_way']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    sdc_track_id           = info['sdc_track_index']   #  the track id of the sdc
    scenario_id            = info['scenario_id']
    
    # We select the 11th timestep as the current timestep to predict the future trajectory of the agents
    curr_t = 11

    all_agent_current = all_agent_trajs[:,curr_t]
    
    
    tracks        = info['predict_list']   # the list of agent ids to predict, the agent id is the index in the all_agent_trajs
    tracks # select # agents
    agents_to_predict = all_agent_trajs[tracks] # # agents
    agent_tp_curr_state_lst = agents_to_predict[:,curr_t]
    
    sdc_agent = all_agent_trajs[sdc_track_id] # one agent
    sdc_agent_curr_state = sdc_agent[curr_t]

    pred_name_lst =["3", "5", "8"]

    ADE_per_scenario, FDE_per_scenario = cvm_pred_pre_scenario(road_polylines,                                                        
                                                                all_agent_current, agent_tp_curr_state_lst,
                                                                sdc_agent_curr_state,
                                                                agents_to_predict,
                                                                sdc_agent,
                                                                curr_t,
                                                                scenario_id)
    for i in range(len(pred_name_lst)):
        ADE_CVM_lst[i].append(ADE_per_scenario[i])
        FDE_CVM_lst[i].append(FDE_per_scenario[i])   

100%|██████████| 287/287 [00:55<00:00,  5.20it/s]


#### 2.3.3 Final ADE & FDE Results

ADE| ADE(3s) | ADE(5s) | ADE(8s) |
FDE| FDE(3s) | FDE(5s) | FDE(8s) |

(Across all target agents and scenarios)

In [7]:
ADE_output = []
FDE_output = []
for i, name in enumerate(pred_name_lst):
    ADE_output.append(f"ADE({name})={round(sum(ADE_CVM_lst[i])/len(ADE_CVM_lst[i]),4)}")
    FDE_output.append(f"FDE({name})={round(sum(FDE_CVM_lst[i])/len(FDE_CVM_lst[i]),4)}")

print("Results of Constant Velocity Model,")
print(f"ADE averaged across all target agents and scenarios:")
print(f"{ADE_output[0]},{ADE_output[1]},{ADE_output[2]}.\n")
print(f"FDE averaged across all target agents and scenarios:")
print(f"{FDE_output[0]},{FDE_output[1]},{FDE_output[2]}.")    

Results of Constant Velocity Model,
ADE averaged across all target agents and scenarios:
ADE(3)=1.4032,ADE(5)=3.6907,ADE(8)=7.8303.

FDE averaged across all target agents and scenarios:
FDE(3)=3.8956,FDE(5)=10.6748,FDE(8)=23.0422.


### 2.4 Q2: Constant Acceleration Model Prediction

Use a constant acceleration model to predict the future trajectories of the target agents at 3s, 5s, and 8s. Then, calculate the Average Displacement Error (ADE) and Final Displacement Error (FDE). The results should be averaged across all target agents and scenarios.

#### 2.4.1 Necessary function definitions

In [30]:
def cam_pred_pre_agent(input_agent, curr_t, type):
    pred_name_lst =["3", "5", "8"]
    color_lst1 = ["darksalmon", "coral", "orangered"]
    color_lst2 = ["lightsteelblue", "cornflowerblue", "royalblue"]

    if type == "sdc_agent":
        color_lst = color_lst1
    elif type == "agents_tp":
        color_lst = color_lst2


    val_t = curr_t
    while is_state_valid(input_agent[val_t]) != True or is_state_valid(input_agent[val_t-1]) != True:
        val_t +=1
    input_agent_curr_state = input_agent[val_t]
    input_agent_pre_state = input_agent[val_t-1]

    # print(val_t)

    agent_a_pred_dic = {}
    for i, name in enumerate(pred_name_lst[::-1]):
        pred = constant_a_pred(input_agent_pre_state, input_agent_curr_state,int(name))
        draw_traj(pred, color_lst[i], f"{type}-{name}s")
        
        predict_horizon = int(int(name)*10)
        gt = input_agent[val_t:min(val_t+predict_horizon, len(input_agent))]

        ADE_score = round(calculate_ADE(pred,gt), 4)
        end_id = -1
        while gt[end_id][9] == 0.:
            end_id -=1
        FDE_score = round(calculate_FDE(pred[end_id],gt[end_id]),4)
        

        # print(f"{name}s: ADE={ADE_score}, FDE={FDE_score}")
            
        agent_a_pred_dic[name] = [pred, ADE_score, FDE_score]
    return agent_a_pred_dic


def cam_pred_pre_scenario(road_polylines,
                          all_agent_current, agents_to_predict_current_state,
                          sdc_current_state,
                          agents_to_predict,
                          sdc_agent,
                          curr_t,
                          scenario_id):
    plt.figure(figsize=(8,8))
    ax = plt.gca()
    fig = plt.gcf()
    fig.set_facecolor('xkcd:grey') 
    ax.set_facecolor('xkcd:grey')


    draw_road(road_polylines)

    for agent in all_agent_current:
        draw_rect(agent)

    for agent in agents_to_predict_current_state:
        draw_rect(agent,"agents_tp")

    
    for agent in agents_to_predict:
        agent_gt = agent[curr_t:curr_t+80]
        draw_gt_traj(agent_gt, "gold", "gt")

    agents_pred_lst = []
    for agent in agents_to_predict:
        agents_pred_lst.append(cam_pred_pre_agent(agent, curr_t, "agents_tp"))


    ax.axis([-70+ sdc_current_state[0], 70+ sdc_current_state[0], -70+ sdc_current_state[1], 70 + sdc_current_state[1]])

    pred_name_lst =["3", "5", "8"]

    ADE_lst = np.array([0,0,0])
    FDE_lst = np.array([0,0,0])
    for agent_dic in agents_pred_lst:
        for i, name in enumerate(pred_name_lst):
            ADE_lst[i] += agent_dic[name][1]
            FDE_lst[i] += agent_dic[name][2]
    ADE_per_scenario = ADE_lst/len(agents_pred_lst)
    FDE_per_scenario = FDE_lst/len(agents_pred_lst)

    
    output_lst = []
    for i, name in enumerate(pred_name_lst):
        output_lst.append(f"{name}s: ADE={round(ADE_per_scenario[i],4)}, FDE={round(FDE_per_scenario[i],4)}")

    

    plt.figtext(0.75, 0.1, f"Average result of agents_tp in this scenario,\n"
                f"{output_lst[0]};\n"
                f"{output_lst[1]};\n"
                f"{output_lst[2]}.",
                ha='center',fontsize=10, bbox=dict(facecolor='white', alpha=0.5, edgecolor='gray'))
    
    plt.subplots_adjust(bottom=0.25)
    
    hands, las = ax.get_legend_handles_labels()
    hands_n = hands[::-1][:3]
    las_n = las[::-1][:3]
    hands_n.append(hands[0])
    las_n.append(las[0])

    ax.legend(hands_n,las_n,loc="upper left")

    plt.title("Constant Acceleration Model Trajectory Prediction")
    filename = f'visualization/CAM/{scenario_id}.png' 
    plt.savefig(filename)       
    plt.close()

    return (ADE_per_scenario, FDE_per_scenario)


#### 2.4.2 Visualisation

Note:
- All 287 scenarios will be stored in png images in folder `task2_prediction/visualization/CAM`.
- In each image,
    - `agents_to_predict` are represented in green blocks, whose predicted trajectories are drawn in blue line gradually becoming lighter over time and the ground truth trajectories of both the above agents are represented in gold dotted line.
    - Other agents are represented in gray blocks.
    - The ADE & FDE scores across all predicted trajectories of `agents_to_predict` w.r.t 3s, 5s, 8s are shown in the right bottom of the image.

In [31]:
# Code for Q2
ADE_CAM_lst = [[],[],[]]
FDE_CAM_lst = [[],[],[]]

for file in tqdm(os.listdir(data_path)):

    info = np.load(data_path+'/'+file,allow_pickle=True)
    all_agent_trajs        = info['all_agent']    
    
    obj_types              = info['object_type']
    lane_polylines         = info['lane']           #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    road_polylines         = info['road_polylines'] #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    crosswalk_polylines    = info['crosswalk']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    speed_bump_polylines   = info['speed_bump']     #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    stop_signs_polylines   = info['stop_sign']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    drive_way_polylines    = info['drive_way']      #  list of [n,7] array [x,y,z,ori_x,ori_y,ori_z,type]
    sdc_track_id           = info['sdc_track_index']   #  the track id of the sdc
    scenario_id            = info['scenario_id']
    
    # We select the 11th timestep as the current timestep to predict the future trajectory of the agents
    curr_t = 11

    all_agent_current = all_agent_trajs[:,curr_t]
    
    
    tracks        = info['predict_list']   # the list of agent ids to predict, the agent id is the index in the all_agent_trajs
    tracks # select # agents
    agents_to_predict = all_agent_trajs[tracks] # # agents
    agent_tp_curr_state_lst = agents_to_predict[:,curr_t]
    
    sdc_agent = all_agent_trajs[sdc_track_id] # one agent
    sdc_agent_curr_state = sdc_agent[curr_t]

    pred_name_lst =["3", "5", "8"]

    ADE_per_scenario, FDE_per_scenario = cam_pred_pre_scenario(road_polylines,                                                        
                                                                all_agent_current, agent_tp_curr_state_lst,
                                                                sdc_agent_curr_state,
                                                                agents_to_predict,
                                                                sdc_agent,
                                                                curr_t,
                                                                scenario_id)
    for i in range(len(pred_name_lst)):
        ADE_CAM_lst[i].append(ADE_per_scenario[i])
        FDE_CAM_lst[i].append(FDE_per_scenario[i])   

100%|██████████| 287/287 [00:54<00:00,  5.31it/s]


#### 2.4.3 Final ADE & FDE Results

ADE| ADE(3s) | ADE(5s) | ADE(8s) |
FDE| FDE(3s) | FDE(5s) | FDE(8s) |

(Across all target agents and scenarios)

In [32]:
ADE_output = []
FDE_output = []
for i, name in enumerate(pred_name_lst):
    ADE_output.append(f"ADE({name})={round(sum(ADE_CAM_lst[i])/len(ADE_CAM_lst[i]),4)}")
    FDE_output.append(f"FDE({name})={round(sum(FDE_CAM_lst[i])/len(FDE_CAM_lst[i]),4)}")

print("Results of Constant Acceleration Model,")
print(f"ADE averaged across all target agents and scenarios:")
print(f"{ADE_output[0]},{ADE_output[1]},{ADE_output[2]}.\n")
print(f"FDE averaged across all target agents and scenarios:")
print(f"{FDE_output[0]},{FDE_output[1]},{FDE_output[2]}.") 

Results of Constant Acceleration Model,
ADE averaged across all target agents and scenarios:
ADE(3)=4.2483,ADE(5)=11.875,ADE(8)=26.7682.

FDE averaged across all target agents and scenarios:
FDE(3)=12.4997,FDE(5)=35.5642,FDE(8)=83.9731.


### 2.5 Discussion

Briefly discuss the observations from your results. 

- Full prediction illustrations can be seen respectively in folder `task2_prediction/visualization/CVM` and folder `task2_prediction/visualization/CAM`, please check!
- Each of the results is within a certain reasonable range of error.
- Eliminating unvalid trajectories of ground truth data greatly improved the ADE & FDE results.